# 05 — Dataset Exploration
Validate schema quality and inspect learnability before model training.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
df = pd.read_csv(PROJECT_ROOT / "data" / "synthetic" / "synthetic_snapshots.csv")
print("rows", len(df), "snapshots", df.snapshot_id.nunique(), "scenarios", df.scenario_id.nunique())


In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
print("Top missing ratios:
", missing.head(10))

print("Scenario distribution:
", df.groupby("scenario_id").snapshot_id.nunique().describe())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12,3))
axes[0].hist(df.current_speed_kmh, bins=40)
axes[0].set_title("Current speed")
axes[1].hist(df.current_density, bins=40)
axes[1].set_title("Current density")
axes[2].hist(df.target_travel_time_sec, bins=40)
axes[2].set_title("Target travel time")
plt.tight_layout()
plt.show()


In [ ]:
corr = df[["current_speed_kmh", "current_density", "queue_length", "capacity_factor", "target_travel_time_sec"]].corr()
print(corr)

sample = df.sample(4000, random_state=42)
plt.figure(figsize=(4,3))
plt.scatter(sample.current_speed_kmh, sample.target_travel_time_sec, s=5, alpha=0.3)
plt.xlabel("current_speed_kmh")
plt.ylabel("target_travel_time_sec")
plt.title("Feature-target relation")
plt.tight_layout()
plt.show()


## Leakage checks
- Confirm `target_travel_time_sec` is not embedded into input features.
- Split by `scenario_id` in Notebook 08.
- Stop and fix schema if any alignment issues are detected.


In [ ]:
assert (df.prediction_horizon_sec == 300).all(), "Unexpected prediction horizon"
assert df.target_travel_time_sec.min() > 0, "Non-positive targets found"
print("Basic schema checks passed")
